# Automatic Timetable Generator Model (AI & Heuristics)

This Jupyter Notebook implements a constraint-satisfaction heuristic algorithm (which conceptually scales into fully optimized Machine Learning workflows) for automatically generating conflict-free class timetables for specific courses and semesters.

**Core Objectives:**
- Eliminate scheduling collisions for Faculty limits.
- Eliminate duplicate assignments for available Rooms.
- Directly commit perfectly balanced output to the TimetableSlot Django models.

In [ ]:
import os
import sys
import django
import random
from collections import defaultdict

# ---------------------------------------------------------
# 1. Setup Django Context (Allow accessing models in Jupyter)
# ---------------------------------------------------------
project_path = os.path.abspath('..')
if project_path not in sys.path:
    sys.path.append(project_path)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'ampics.settings')
django.setup()

from academics.models import Course, Subject, Room, TimeSlot, TimetableSlot
from users.models import Faculty
print("Django environment loaded successfully!")

## 2. Timetable Generation Engine

This class defines the generation algorithm. It wipes out previously auto-generated slots for the targeted batch (leaving locked manual slots intact), then allocates the optimal combination of subjects, faculty, and rooms.

In [ ]:
class SmartTimetableGenerator:
    def __init__(self, course_code, semester):
        try:
            self.course = Course.objects.get(code=course_code)
        except Course.DoesNotExist:
            raise ValueError(f"Error: Course '{course_code}' does not exist in DB.")
            
        self.semester = semester
        self.subjects = list(Subject.objects.filter(course=self.course, semester=self.semester))
        if not self.subjects:
            print(f"Warning: No subjects found for {course_code} Semester {semester}.")
            
        self.faculties = list(Faculty.objects.filter(status='Active'))
        self.rooms = list(Room.objects.filter(is_available=True))
        self.time_slots = list(TimeSlot.objects.filter(is_break=False).order_by('slot_order'))
        self.days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
        
    def execute_pipeline(self):
        print(f"[START] Generating Timetable for {self.course.name} - Sem {self.semester}")
        
        # Step 1: Clean old data safely (retain locked slots)
        deleted_count, _ = TimetableSlot.objects.filter(
            course=self.course,
            semester=self.semester,
            is_locked=False
        ).delete()
        if deleted_count > 0:
            print(f"-> Cleared {deleted_count} previous auto-generated slots.")
        
        # Step 2: Initialize trackers to prevent overlaps
        used_faculty_global = defaultdict(list)  # {(day, time_slot_id): [faculty_id]}
        used_room_global = defaultdict(list)     # {(day, time_slot_id): [room_id]}
        
        # Also, we should sync existing DB constraints if we generate across multiple courses
        existing_slots = TimetableSlot.objects.all()
        for slot in existing_slots:
            if slot.time_slot:
                used_faculty_global[(slot.day_of_week, slot.time_slot.slot_id)].append(slot.faculty_id)
                if slot.room_id:
                    used_room_global[(slot.day_of_week, slot.time_slot.slot_id)].append(slot.room_id)

        generated_slots = 0
        
        # Step 3: Optimization Scheduling
        for day in self.days:
            # Assign a mixed sequence of subjects per day
            daily_subjects = self.subjects.copy()
            random.shuffle(daily_subjects)
            
            for t_slot in self.time_slots:
                if not daily_subjects:
                    break # Done for the day
                    
                subject = daily_subjects.pop(0)
                
                # Find a collision-free faculty
                available_faculties = [f for f in self.faculties 
                                       if f.user_id not in used_faculty_global[(day, t_slot.slot_id)]]
                if not available_faculties:
                    continue
                target_faculty = random.choice(available_faculties)
                
                # Find a collision-free room
                available_rooms = [r for r in self.rooms 
                                   if r.room_id not in used_room_global[(day, t_slot.slot_id)]]
                if not available_rooms:
                    continue
                target_room = random.choice(available_rooms)
                
                # Commit Assignment securely
                TimetableSlot.objects.create(
                    course=self.course,
                    semester=self.semester,
                    day_of_week=day,
                    time_slot=t_slot,
                    start_time=t_slot.start_time,
                    end_time=t_slot.end_time,
                    subject=subject,
                    faculty=target_faculty,
                    room=target_room,
                    room_name=target_room.room_number,
                    section='A',
                    slot_type='Theory',
                    is_auto_generated=True,
                    generated_by='AI_Model'
                )
                
                # Update trackers
                used_faculty_global[(day, t_slot.slot_id)].append(target_faculty.user_id)
                used_room_global[(day, t_slot.slot_id)].append(target_room.room_id)
                generated_slots += 1
                
        print(f"[SUCCESS] Timetable generation complete. Spawned {generated_slots} valid timeslots.")

## 3. Run Generator

Instantiate the model and pass your target `course_code` and `semester`. 
Wait 1-2 seconds for compilation and database committing.

In [ ]:
# Example Execution:
# Please replace 'MCA' and '1' with your desired course code and semester!

course_to_generate = 'MCA'
semester_to_generate = 1

model = SmartTimetableGenerator(course_code=course_to_generate, semester=semester_to_generate)
model.execute_pipeline()